# 나이브 베이즈 알고리즘
나이브 베이즈는 “과거에 스팸으로 분류된 메일에서 이런 단어들이 자주 나타났으므로, 현재 메일도 스팸일 가능성이 높다”라고 판단하는 알고리즘이다.
```
과거 메일과 분류 결과
        ↓
스팸·정상에서 자주 등장하는 단어 학습
        ↓
새 메일의 단어 확인
        ↓
스팸일 확률과 정상일 확률 계산
        ↓
확률이 더 높은 클래스로 분류
```

# 특정 단어로 스팸 메일 분류하기

메일에 어떤 단어가 들어 있는지를 이용해 **스팸**과 **정상**을 분류하기

| 스팸 메일에 자주 등장하는 단어 | 정상 메일에 자주 등장하는 단어 |
|---|---|
| 무료, 당첨, 경품, 대출, 광고 | 회의, 보고서, 프로젝트, 일정, 자료 |

## 1. 베이즈 정리를 메일 데이터로 계산하기

과거에 분류한 메일이 10개 있다고 가정한다.

| 메일 종류 | 전체 메일 | `무료` 포함 | `무료` 미포함 |
|---|---:|---:|---:|
| 스팸 | 4 | 3 | 1 |
| 정상 | 6 | 1 | 5 |
| 합계 | 10 | 4 | 6 |

### 1단계: 메일을 보기 전

10개 중 스팸이 4개이므로 아무 단어도 확인하지 않았을 때 스팸일 확률은 다음과 같다. 이를 **사전확률**이라고 한다.

$$P(\text{스팸})=\frac{4}{10}=0.4$$

### 2단계: `무료`라는 단어 발견

스팸 4개 중 3개에 `무료`가 들어 있다. 따라서 스팸일 때 `무료`가 나타날 확률인 **가능도**는 다음과 같다.

$$P(\text{무료}\mid\text{스팸})=\frac{3}{4}=0.75$$

전체 메일에서는 10개 중 4개에 `무료`가 들어 있다.

$$P(\text{무료})=\frac{4}{10}=0.4$$

### 3단계: `무료`를 발견한 뒤 스팸일 확률

베이즈 정리에 앞의 값을 넣는다.

$$P(\text{스팸}\mid\text{무료})=\frac{P(\text{무료}\mid\text{스팸})P(\text{스팸})}{P(\text{무료})}$$

$$P(\text{스팸}\mid\text{무료})=\frac{0.75\times0.4}{0.4}=0.75$$

즉, `무료`가 포함된 메일 4개 중 3개가 스팸이므로 새 메일에서 `무료`를 발견하면 스팸일 확률을 **75%**로 판단할 수 있다.

| 확률 기호 | 데이터에서의 의미 | 계산값 |
|:---|:---|---:|
| $P(\text{스팸})$ | 단어를 보기 전 스팸 확률 | 0.40 |
| $P(\text{무료}\mid\text{스팸})$ | 스팸에서 `무료`가 나올 확률 | 0.75 |
| $P(\text{무료})$ | 전체 메일에서 `무료`가 나올 확률 | 0.40 |
| $P(\text{스팸}\mid\text{무료})$ | `무료`를 본 뒤 스팸 확률 | 0.75 |

In [1]:
# 위의 계산을 Python으로 확인
p_spam = 4 / 10
p_free_given_spam = 3 / 4
p_free = 4 / 10

p_spam_given_free = p_free_given_spam * p_spam / p_free
print(f"'무료'가 포함된 메일이 스팸일 확률: {p_spam_given_free:.0%}")

'무료'가 포함된 메일이 스팸일 확률: 75%


## 베이즈 정리에서 나이브 베이즈로

실제 메일에는 `무료` 하나만 있는 것이 아니라 `무료`, `당첨`, `경품`처럼 여러 단어가 함께 등장한다.  
나이브 베이즈는 스팸 또는 정상이라는 클래스가 정해졌을 때 각 단어가 서로 독립적으로 등장한다고 단순하게 가정한다.  

$$P(\text{스팸}\mid\text{무료},\text{당첨},\text{경품}) \propto P(\text{스팸})\times P(\text{무료}\mid\text{스팸})\times P(\text{당첨}\mid\text{스팸})\times P(\text{경품}\mid\text{스팸})$$

같은 방식으로 정상 메일의 점수도 계산한 뒤 더 큰 쪽으로 분류한다.

```text
'무료 경품에 당첨되었습니다'
               ↓
무료·경품·당첨의 클래스별 출현 확률 계산
               ↓
스팸 점수와 정상 점수 비교
               ↓
스팸 점수가 더 크면 '스팸'으로 분류
```

이제 아래 실습에서 `CountVectorizer`가 단어를 세고, `MultinomialNB`가 이 확률 계산을 수행한다.

## 1. 학습용 메일 준비

컴퓨터에게 스팸 메일과 정상 메일의 예시를 각각 보여준다.

In [5]:
train_mails = [
    '무료 경품에 당첨되었습니다',
    '지금 대출을 신청하세요',
    '무료 쿠폰 광고입니다',
    '당첨 상품을 받아가세요',
    '내일 프로젝트 회의가 있습니다',
    '주간 업무 보고서를 제출하세요',
    '회의 일정과 자료를 공유합니다',
    '프로젝트 결과를 보고합니다'
]

# 1은 스팸, 0은 정상
y = [1, 1, 1, 1, 0, 0, 0, 0]

## 2. 단어를 숫자로 바꾸고 모델 학습

`CountVectorizer`는 각 메일에 단어가 몇 번 등장했는지 센다. `MultinomialNB`는 스팸과 정상에서 어떤 단어가 자주 나오는지 학습한다.

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import numpy as np

# ① 메일의 단어를 숫자로 변환
vectorizer = CountVectorizer()
vectorizer

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [16]:
vectorizer.fit(train_mails)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [17]:
X_train = vectorizer.transform(train_mails)
X_train.toarray()[:5]

array([[0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        1, 0, 1]])

In [19]:
# ② 나이브 베이즈 모델 학습
model = MultinomialNB()
model.fit(X_train, y)

model

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[4.,4.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.69,-0.69]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 25)","[[1.,0.,1.,...,2.,1.,1.], [0.,1.,0.,...,0.,0.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 25)","[[-3. ,-3.69,-3. ,...,-2.59,-3. ,-3. ], [-3.61,-2.92,-3.61,...,-3.61,-3.61,-3.61]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,25


In [20]:
print('학습한 단어:', vectorizer.get_feature_names_out())

학습한 단어: ['결과를' '경품에' '공유합니다' '광고입니다' '내일' '당첨' '당첨되었습니다' '대출을' '무료' '받아가세요' '보고서를'
 '보고합니다' '상품을' '신청하세요' '업무' '일정과' '있습니다' '자료를' '제출하세요' '주간' '지금' '쿠폰'
 '프로젝트' '회의' '회의가']


## 3. 새로운 메일 분류

학습한 모델에 새로운 메일을 입력한다. 결과가 1이면 스팸, 0이면 정상이다.

In [21]:
new_mails = [
    '무료 경품에 당첨되었습니다',
    '프로젝트 회의 자료를 공유합니다',
    '무료 대출 광고입니다'
]

X_new = vectorizer.transform(new_mails)
predictions = model.predict(X_new)
spam_probabilities = model.predict_proba(X_new)[:, 1]

for mail, prediction, probability in zip(new_mails, predictions, spam_probabilities):
    result = '스팸' if prediction == 1 else '정상'
    print(f'{mail} → {result} (스팸 확률: {probability:.2f})')

무료 경품에 당첨되었습니다 → 스팸 (스팸 확률: 0.94)
프로젝트 회의 자료를 공유합니다 → 정상 (스팸 확률: 0.05)
무료 대출 광고입니다 → 스팸 (스팸 확률: 0.88)


## 핵심 정리

```text
무료·당첨·대출·광고가 등장한 학습 메일 → 스팸
회의·보고서·프로젝트·일정이 등장한 학습 메일 → 정상
                         ↓
새 메일에 등장한 단어를 보고 스팸 확률 계산
```

특정 단어를 조건문으로 직접 지정한 것이 아니라,   
나이브 베이즈 모델이 학습 메일에서 **클래스별 단어 출현 확률**을 계산하여 분류한다.